<a href="https://colab.research.google.com/github/iav2002/AppliedDeepLearning/blob/main/Part2_NetworkVariations_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part 2, Notebook 5, Network Depth and Skip Connections

In this notebook I investigate two coupled architectural choices, network depth (how many conv blocks before the FC head) and skip connections (ResNet-style residuals). They're paired in one notebook because deep networks without skip connections suffer vanishing gradients, so the cleanest experiment is to compare deep variants with and without skips against the baseline.

This time I run on both tasks. Depth interacts with overfitting (more parameters, easier to overfit), and the two tasks behave very differently on that front (regression trained clean across 20 epochs while classification overfit at epoch 7). So the same architectural change can lead to genuinely different conclusions, which is worth seeing.

Reduced the amount of epochs to experiment faster

## 1. Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!cp "/content/drive/MyDrive/Colab Notebooks/AppliedDL/face_age.zip" /content/
!cp -r "/content/drive/MyDrive/Colab Notebooks/AppliedDL/data_splits" /content/
!unzip -q /content/face_age.zip -d /content/

## 2. Imports

Same as notebook 4.



In [3]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from pathlib import Path

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

device: cuda
gpu: NVIDIA L4


## 3. Dataset class

Same `FaceAgeDataset` carried over.

In [4]:
CATEGORIES = ["infant", "child", "teen", "youth", "mid", "mature", "senior"]
CAT_TO_IDX = {c: i for i, c in enumerate(CATEGORIES)}


class FaceAgeDataset(Dataset):
    def __init__(self, csv_path, task, transform=None):
        self.df = pd.read_csv(csv_path)
        self.task = task
        self.transform = transform
        self.paths = ["/content/" + p for p in self.df["path"]]

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        if self.transform:
            img = self.transform(img)

        if self.task == "regression":
            label = torch.tensor(self.df.iloc[idx]["age"], dtype=torch.float32)
        else:
            cat = self.df.iloc[idx]["age_category"]
            label = torch.tensor(CAT_TO_IDX[cat], dtype=torch.long)

        return img, label

## 4. Transforms and dataloaders

Same minimal pipeline, both tasks active this notebook.

In [5]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

train_ds_reg = FaceAgeDataset("/content/data_splits/train.csv", "regression", transform)
val_ds_reg   = FaceAgeDataset("/content/data_splits/val.csv",   "regression", transform)

train_ds_cls = FaceAgeDataset("/content/data_splits/train.csv", "classification", transform)
val_ds_cls   = FaceAgeDataset("/content/data_splits/val.csv",   "classification", transform)

BATCH_SIZE = 64

train_loader_reg = DataLoader(train_ds_reg, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=2, pin_memory=True, persistent_workers=True)
val_loader_reg   = DataLoader(val_ds_reg,   batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=2, pin_memory=True, persistent_workers=True)

train_loader_cls = DataLoader(train_ds_cls, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=2, pin_memory=True, persistent_workers=True)
val_loader_cls   = DataLoader(val_ds_cls,   batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=2, pin_memory=True, persistent_workers=True)

print(f"train: {len(train_ds_reg)}    val: {len(val_ds_reg)}")

train: 7320    val: 1464


## 5. Train and eval functions


In [6]:
def train_one_epoch(model, loader, loss_fn, optimizer):
    model.train()
    total_loss = 0
    n_samples = 0

    for imgs, lbls in loader:
        imgs, lbls = imgs.to(device), lbls.to(device)

        out = model(imgs)
        if out.shape[-1] == 1:
            out = out.squeeze(-1)
        loss = loss_fn(out, lbls)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * imgs.size(0)
        n_samples += imgs.size(0)

    return total_loss / n_samples


def evaluate(model, loader, loss_fn, task):
    model.eval()
    total_loss = 0
    n_samples = 0
    correct = 0
    abs_error_sum = 0

    with torch.no_grad():
        for imgs, lbls in loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            out = model(imgs)
            if out.shape[-1] == 1:
                out = out.squeeze(-1)
            loss = loss_fn(out, lbls)

            total_loss += loss.item() * imgs.size(0)
            n_samples += imgs.size(0)

            if task == "regression":
                abs_error_sum += (out - lbls).abs().sum().item()
            else:
                preds = out.argmax(dim=1)
                correct += (preds == lbls).sum().item()

    avg_loss = total_loss / n_samples
    if task == "regression":
        metric = abs_error_sum / n_samples
    else:
        metric = correct / n_samples

    return avg_loss, metric

## 6. Variant runner

Same wrapper as notebook 4, defaults tightened to max 12 epochs and patience 3 for faster sweeps.

In [7]:
def run_variant(model, train_loader, val_loader, loss_fn, task,
                max_epochs=12, patience=3, lr=1e-3, verbose=True):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    if task == "regression":
        best_metric = float("inf")
        better = lambda new, best: new < best
    else:
        best_metric = -float("inf")
        better = lambda new, best: new > best

    history = {"train_loss": [], "val_loss": [], "val_metric": []}
    best_state = None
    epochs_no_improve = 0

    for epoch in range(1, max_epochs + 1):
        train_loss = train_one_epoch(model, train_loader, loss_fn, optimizer)
        val_loss, val_metric = evaluate(model, val_loader, loss_fn, task)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_metric"].append(val_metric)

        if better(val_metric, best_metric):
            best_metric = val_metric
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
            marker = " (best)"
        else:
            epochs_no_improve += 1
            marker = ""

        if verbose:
            print(f"  epoch {epoch:2d}    train_loss {train_loss:7.4f}    "
                  f"val_loss {val_loss:7.4f}    val_metric {val_metric:.4f}{marker}")

        if epochs_no_improve >= patience:
            if verbose:
                print(f"  stopped early at epoch {epoch}, no improvement for {patience} epochs")
            break

    model.load_state_dict(best_state)
    return {"best_metric": best_metric, "best_state": best_state, "history": history}